# Fine-tuning DistilBERT (Deep-LLM)

Ce notebook fine-tune **DistilBERT** pré-entraîné pour la classification de sentiment des avis Yelp (1-5 étoiles).

**Grille** : *Deep-LLM* → 1 pt | *Arch Deep: plusieurs* → 4 pts

### Checklist SAE-119
- [ ] Charger DistilBERT pré-entraîné + tokenizer
- [ ] Tokeniser les textes avec DistilBertTokenizer
- [ ] Ajouter une couche de classification sur DistilBERT
- [ ] Fine-tuning avec learning rate 2e-5, 3-5 epochs
- [ ] Métriques : accuracy, precision, recall, f1, confusion matrix
- [ ] Sauvegarder le modèle fine-tuné
- [ ] Notebook exécutable sans erreur

## 0. Imports et Configuration

> **Prérequis** : `pip install torch transformers`

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import DistilBertTokenizer, DistilBertModel

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, classification_report, confusion_matrix
)

import warnings
warnings.filterwarnings('ignore')

# Chemins
DATA_DIR   = '../../data/cleaned'
MODELS_DIR = '../../models/'
os.makedirs(MODELS_DIR, exist_ok=True)

# Device
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device         : {DEVICE}')
print(f'PyTorch version: {torch.__version__}')
import transformers
print(f'Transformers   : {transformers.__version__}')

## 1. Chargement des Données

In [ ]:
print('Chargement des données...')
try:
    df = pd.read_parquet(os.path.join(DATA_DIR, 'reviews_clean.parquet'), engine='fastparquet')
except Exception:
    df = pd.read_parquet(os.path.join(DATA_DIR, 'reviews_clean.parquet'))

df = df.dropna(subset=['text', 'stars'])
print(f'Dimensions du dataset : {df.shape}')
print(f"Distribution des étoiles :\n{df['stars'].value_counts(normalize=True).sort_index()}")

In [ ]:
# Échantillonnage réduit pour DistilBERT (lourd à entraîner)
# Augmenter SAMPLE_SIZE si vous disposez d'un GPU
SAMPLE_SIZE = 2000  # CPU: ~2000 | GPU: 5000-10000

df_sample = df.sample(n=SAMPLE_SIZE, random_state=42)

# Labels 1-5 → 0-4 pour PyTorch
texts  = df_sample['text'].tolist()
labels = (df_sample['stars'].astype(int) - 1).tolist()

# Split 80/10/10 — identique aux autres notebooks
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    texts, labels, test_size=0.20, random_state=42, stratify=labels
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

print(f'Train : {len(X_train)} | Val : {len(X_val)} | Test : {len(X_test)}')

## 2. Tokenisation avec DistilBertTokenizer

In [ ]:
MODEL_NAME  = 'distilbert-base-uncased'
MAX_LENGTH  = 128   # Réduire pour accélérer sur CPU (256 sur GPU)

print(f'Chargement du tokenizer : {MODEL_NAME}...')
tokenizer = DistilBertTokenizer.from_pretrained(MODEL_NAME)
print('Tokenizer chargé.')

In [ ]:
class YelpDataset(Dataset):
    """Dataset PyTorch pour les avis Yelp + DistilBERT tokenizer."""
    
    def __init__(self, texts, labels, tokenizer, max_length):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors='pt'
        )
        self.labels = torch.tensor(labels, dtype=torch.long)
    
    def __len__(self):
        return len(self.labels)
    
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item


BATCH_SIZE = 16  # Réduire à 8 si OOM sur CPU

train_dataset = YelpDataset(X_train, y_train, tokenizer, MAX_LENGTH)
val_dataset   = YelpDataset(X_val,   y_val,   tokenizer, MAX_LENGTH)
test_dataset  = YelpDataset(X_test,  y_test,  tokenizer, MAX_LENGTH)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f'Batches train: {len(train_loader)} | val: {len(val_loader)} | test: {len(test_loader)}')

## 3. Modèle : DistilBERT + Couche de Classification

Architecture :
```
DistilBERT (pré-entraîné, fine-tunable)
  ↓ [CLS] token hidden state (dim=768)
Dropout(0.3)
Linear(768 → 256) + ReLU
Dropout(0.3)
Linear(256 → 5)   ← classification 5 classes
```

In [ ]:
class DistilBertClassifier(nn.Module):
    """DistilBERT avec une tête de classification custom."""
    
    def __init__(self, model_name: str, num_classes: int = 5, dropout: float = 0.3):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size  # 768
        
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Utiliser le token [CLS] (position 0)
        cls_output = outputs.last_hidden_state[:, 0, :]  # (batch, 768)
        return self.classifier(cls_output)


print(f'Chargement de {MODEL_NAME}...')
model = DistilBertClassifier(MODEL_NAME, num_classes=5).to(DEVICE)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Paramètres totaux      : {total_params:,}')
print(f'Paramètres entraînable : {trainable_params:,}')

## 4. Fine-tuning (Adam lr=2e-5 + CrossEntropyLoss)

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        y_batch        = batch['labels'].to(device)
        
        optimizer.zero_grad()
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, y_batch)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item() * len(y_batch)
        correct    += (logits.argmax(1) == y_batch).sum().item()
        total      += len(y_batch)
    return total_loss / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for batch in loader:
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        y_batch        = batch['labels'].to(device)
        
        logits = model(input_ids, attention_mask)
        loss   = criterion(logits, y_batch)
        
        total_loss += loss.item() * len(y_batch)
        correct    += (logits.argmax(1) == y_batch).sum().item()
        total      += len(y_batch)
    return total_loss / total, correct / total

In [ ]:
# Hyperparamètres — identiques à la checklist
EPOCHS   = 3      # 3-5 epochs selon la checklist (3 pour CPU, 5 sur GPU)
LR       = 2e-5   # Learning rate recommandé pour BERT
PATIENCE = 2

criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)

history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': []}
best_val_loss = float('inf')
best_model_path = os.path.join(MODELS_DIR, 'distilbert_best.pt')
patience_counter = 0

print(f'Entraînement sur {DEVICE} pendant {EPOCHS} epochs...')
print('(Peut prendre plusieurs minutes sur CPU)\n')

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    vl_loss, vl_acc = eval_epoch(model, val_loader, criterion, DEVICE)
    
    history['train_loss'].append(tr_loss)
    history['val_loss'].append(vl_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(vl_acc)
    
    print(f'Epoch {epoch}/{EPOCHS} | '
          f'Train loss: {tr_loss:.4f} acc: {tr_acc:.4f} | '
          f'Val loss: {vl_loss:.4f} acc: {vl_acc:.4f}')
    
    if vl_loss < best_val_loss:
        best_val_loss = vl_loss
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping à l\'epoch {epoch}.')
            break

print('\nEntraînement terminé.')

## 5. Courbes de Loss Train / Val

In [ ]:
epochs_ran = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(epochs_ran, history['train_loss'], label='Train', marker='o')
axes[0].plot(epochs_ran, history['val_loss'],   label='Val',   marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('CrossEntropy Loss')
axes[0].set_title('Loss Train / Validation — DistilBERT')
axes[0].legend()
axes[0].grid(alpha=0.3)

axes[1].plot(epochs_ran, history['train_acc'], label='Train', marker='o', color='darkorange')
axes[1].plot(epochs_ran, history['val_acc'],   label='Val',   marker='s', color='steelblue')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Accuracy Train / Validation — DistilBERT')
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Évaluation sur le Test Set

In [ ]:
# Recharger les meilleurs poids
model.load_state_dict(torch.load(best_model_path, map_location=DEVICE))
model.eval()

all_preds, all_labels = [], []
with torch.no_grad():
    for batch in test_loader:
        input_ids      = batch['input_ids'].to(DEVICE)
        attention_mask = batch['attention_mask'].to(DEVICE)
        logits = model(input_ids, attention_mask)
        all_preds.extend(logits.argmax(1).cpu().numpy())
        all_labels.extend(batch['labels'].numpy())

all_preds  = np.array(all_preds)
all_labels = np.array(all_labels)

acc  = accuracy_score(all_labels, all_preds)
prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
rec  = recall_score(all_labels, all_preds,    average='macro', zero_division=0)
f1   = f1_score(all_labels, all_preds,         average='macro', zero_division=0)

print('=== RÉSULTATS SUR LE TEST SET ===')
print(f'Accuracy  : {acc:.4f}')
print(f'Precision : {prec:.4f}')
print(f'Recall    : {rec:.4f}')
print(f'F1 Macro  : {f1:.4f}')

In [ ]:
# Rapport de classification
target_names = [f'{i+1} étoile(s)' for i in range(5)]
print(classification_report(all_labels, all_preds, target_names=target_names, zero_division=0))

In [ ]:
# Matrice de confusion
cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False,
            xticklabels=target_names, yticklabels=target_names)
plt.title('Matrice de Confusion — DistilBERT (Test)')
plt.ylabel('Vrai')
plt.xlabel('Prédit')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. Sauvegarde du Modèle Fine-tuné

In [ ]:
# Sauvegarder le modèle complet (poids + config)
final_model_dir = os.path.join(MODELS_DIR, 'distilbert_finetuned')
os.makedirs(final_model_dir, exist_ok=True)

# Sauvegarder les poids PyTorch
torch.save({
    'model_state_dict': model.state_dict(),
    'model_name':       MODEL_NAME,
    'num_classes':      5,
    'max_length':       MAX_LENGTH,
    'accuracy_test':    float(acc),
    'f1_macro_test':    float(f1)
}, os.path.join(final_model_dir, 'model.pt'))

# Sauvegarder aussi le tokenizer pour faciliter l'inférence
tokenizer.save_pretrained(os.path.join(final_model_dir, 'tokenizer'))

print(f'Modèle sauvegardé dans : {final_model_dir}/')
print(f'  └── model.pt')
print(f'  └── tokenizer/')
print(f'\nPerformance finale (test) | Accuracy: {acc:.4f} | F1 Macro: {f1:.4f}')

---
## ✅ Résumé

| Étape | Résultat |
|-------|----------|
| Modèle de base | `distilbert-base-uncased` (HuggingFace) |
| Tokenisation | `DistilBertTokenizer`, max_length=128 |
| Architecture | DistilBERT → [CLS] → Dropout → Linear(768→256) → Linear(256→5) |
| Optimiseur | AdamW (lr=2e-5, weight_decay=0.01) |
| Epochs | 3 (CPU) — augmenter à 5 sur GPU |
| Modèle sauvegardé | `models/distilbert_finetuned/model.pt` |

> **Note GPU** : Augmenter `SAMPLE_SIZE` à 5000-10000 et `EPOCHS` à 5 si GPU disponible.

**Prérequis** :
```bash
pip install torch transformers
```